# 01. TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING)
## Dự án: Movie Recommender System - Hệ Thống Gợi Ý Phim Cá Nhân Hóa

Mục tiêu của notebook:
1. **Đọc dữ liệu thô (Raw Data Ingestion)**: Đọc 3 tệp dữ liệu từ thư mục `data/raw/` gồm `ratings.dat`, `users.dat`, `movies.dat` (định dạng phân cách `::`, encoding `latin-1`).
2. **Xử lý & Chuẩn hóa thuộc tính (Attribute Normalization)**:
   - Gán và mapping tên cột chuẩn kỹ thuật cho các bảng dữ liệu.
   - Chuẩn hóa thuộc tính `Gender` ('M' -> 'Male', 'F' -> 'Female').
   - Chuẩn hóa và mapping mã `Age` sang nhóm tuổi `age_group` (ví dụ: '18-24', '25-34',...).
   - Chuẩn hóa và mapping mã nghề nghiệp `Occupation` sang tên nghề nghiệp tường minh `occupation_name`.
   - Trích xuất `Release_Year` (năm phát hành) từ tiêu đề `Title` và tạo `clean_title`.
   - Chuẩn hóa chuỗi thể loại phim `Genres`.
   - Chuyển đổi thuộc tính `timestamp` sang định dạng ngày giờ chuẩn `datetime`.
3. **Lưu kết quả (Data Persistence)**: Xuất 3 tệp CSV đã làm sạch vào thư mục `data/processed/`:
   - `cleaned_users.csv`
   - `cleaned_movies.csv`
   - `cleaned_ratings.csv`
4. **Hiển thị kết quả**: Hiển thị 5 dòng đầu tiên (`.head()`) của từng DataFrame sau khi xử lý.

In [1]:
import os
import re
import pandas as pd
import numpy as np

# Cấu hình đường dẫn thư mục linh hoạt
# Hoạt động chính xác khi chạy từ thư mục gốc hoặc thư mục notebooks/
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == 'notebooks':
    BASE_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '..'))
else:
    BASE_DIR = os.path.abspath(CURRENT_DIR)

DATA_RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Raw Data Directory: {DATA_RAW_DIR}")
print(f"Processed Data Directory: {DATA_PROCESSED_DIR}")

Base Directory: G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system
Raw Data Directory: G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system\data\raw
Processed Data Directory: G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system\data\processed


## 1. Đọc dữ liệu thô (Raw Data Ingestion)
Đọc 3 tệp dữ liệu thô MovieLens 1M từ thư mục `data/raw/`:
- `users.dat`: Thông tin người dùng (`UserID::Gender::Age::Occupation::Zip-code`)
- `movies.dat`: Thông tin các bộ phim (`MovieID::Title::Genres`)
- `ratings.dat`: Lịch sử đánh giá (`UserID::MovieID::Rating::Timestamp`)

*Lưu ý: Dữ liệu sử dụng dấu phân cách `::`, không có dòng header và sử dụng bảng mã `latin-1`.*

In [2]:
# Thiết lập đường dẫn các tệp dữ liệu thô
users_path = os.path.join(DATA_RAW_DIR, 'users.dat')
movies_path = os.path.join(DATA_RAW_DIR, 'movies.dat')
ratings_path = os.path.join(DATA_RAW_DIR, 'ratings.dat')

# Đọc users.dat
users_raw = pd.read_csv(
    users_path,
    sep='::',
    engine='python',
    encoding='latin-1',
    header=None,
    names=['user_id', 'gender', 'age', 'occupation', 'zip_code']
)

# Đọc movies.dat
movies_raw = pd.read_csv(
    movies_path,
    sep='::',
    engine='python',
    encoding='latin-1',
    header=None,
    names=['movie_id', 'title', 'genres']
)

# Đọc ratings.dat
ratings_raw = pd.read_csv(
    ratings_path,
    sep='::',
    engine='python',
    encoding='latin-1',
    header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

print(f"Đã đọc users_raw:   {users_raw.shape[0]:,} dòng, {users_raw.shape[1]} cột")
print(f"Đã đọc movies_raw:  {movies_raw.shape[0]:,} dòng, {movies_raw.shape[1]} cột")
print(f"Đã đọc ratings_raw: {ratings_raw.shape[0]:,} dòng, {ratings_raw.shape[1]} cột")

Đã đọc users_raw:   6,040 dòng, 5 cột
Đã đọc movies_raw:  3,883 dòng, 3 cột
Đã đọc ratings_raw: 1,000,209 dòng, 4 cột


## 2. Tiền xử lý & Chuẩn hóa thuộc tính (Attribute Normalization)

### 2.1. Định nghĩa các từ điển ánh xạ (Mappings) theo đặc tả MovieLens 1M
- **Age**: Chuyển đổi mã số tuổi sang dải tuổi mô tả thực tế.
- **Occupation**: Chuyển đổi mã nghề nghiệp từ 0 đến 20 sang danh xưng nghề nghiệp tiếng Anh.
- **Gender**: Ánh xạ ký hiệu giới tính `M` -> `Male`, `F` -> `Female`.

In [3]:
# Ánh xạ nhóm tuổi (Age Mapping)
AGE_MAP = {
    1: "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+"
}

# Ánh xạ ngành nghề (Occupation Mapping)
OCCUPATION_MAP = {
    0: "other",
    1: "academic/educator",
    2: "artist",
    3: "clerical/admin",
    4: "college/grad student",
    5: "customer service",
    6: "doctor/health care",
    7: "executive/managerial",
    8: "farmer",
    9: "homemaker",
    10: "K-12 student",
    11: "lawyer",
    12: "programmer",
    13: "retired",
    14: "sales/marketing",
    15: "scientist",
    16: "self-employed",
    17: "technician/engineer",
    18: "tradesman/craftsman",
    19: "unemployed",
    20: "writer"
}

# Ánh xạ giới tính (Gender Mapping)
GENDER_MAP = {
    'M': 'Male',
    'F': 'Female'
}

### 2.2. Chuẩn hóa DataFrame Users
- Giữ nguyên `gender` ('M'/'F') đồng thời tạo `gender_desc` ('Male'/'Female').
- Giữ nguyên mã số `age` đồng thời tạo cột `age_group` chứa dải tuổi tường minh.
- Giữ nguyên mã số `occupation` đồng thời tạo cột `occupation_name` chứa tên nghề nghiệp.
- Làm sạch định dạng `zip_code`.

In [4]:
cleaned_users = users_raw.copy()

# Chuẩn hóa giới tính (Gender)
cleaned_users['gender_desc'] = cleaned_users['gender'].map(GENDER_MAP)

# Chuẩn hóa nhóm tuổi (Age)
cleaned_users['age_group'] = cleaned_users['age'].map(AGE_MAP)

# Chuẩn hóa ngành nghề (Occupation)
cleaned_users['occupation_name'] = cleaned_users['occupation'].map(OCCUPATION_MAP)

# Làm sạch mã bưu chính (Zip-code)
cleaned_users['zip_code'] = cleaned_users['zip_code'].astype(str).str.strip()

# Sắp xếp lại các cột theo thứ tự hợp lý
cleaned_users = cleaned_users[[
    'user_id', 'gender', 'gender_desc', 'age', 'age_group', 
    'occupation', 'occupation_name', 'zip_code'
]]

print("Chuẩn hóa DataFrame Users thành công!")
print(f"Kích thước sau xử lý: {cleaned_users.shape}")

Chuẩn hóa DataFrame Users thành công!
Kích thước sau xử lý: (6040, 8)


### 2.3. Chuẩn hóa DataFrame Movies
- Trích xuất `release_year` (Năm phát hành) từ cột `title` bằng biểu thức chính quy (Regex: `\((\d{4})\)$`).
- Tạo cột `clean_title` loại bỏ phần `(YYYY)` ở cuối để tên phim tinh gọn.
- Chuẩn hóa khoảng trắng của thuộc tính thể loại `genres`.

In [5]:
cleaned_movies = movies_raw.copy()

# 1. Trích xuất năm phát hành (Release_Year) từ tiêu đề
# Định dạng trong dữ liệu thô: "Tên Phim (YYYY)"
cleaned_movies['release_year'] = cleaned_movies['title'].str.extract(r'\((\d{4})\)\s*$', expand=False).astype(int)

# 2. Làm sạch tiêu đề phim (loại bỏ năm trong ngoặc đơn ở cuối tên phim)
cleaned_movies['clean_title'] = cleaned_movies['title'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True).str.strip()

# 3. Chuẩn hóa cột thể loại (Genres)
cleaned_movies['genres'] = cleaned_movies['genres'].str.strip()

# Sắp xếp lại các cột
cleaned_movies = cleaned_movies[['movie_id', 'title', 'clean_title', 'release_year', 'genres']]

print("Chuẩn hóa DataFrame Movies thành công!")
print(f"Kích thước sau xử lý: {cleaned_movies.shape}")

Chuẩn hóa DataFrame Movies thành công!
Kích thước sau xử lý: (3883, 5)


### 2.4. Chuẩn hóa DataFrame Ratings
- Kiểm tra và đảm bảo đúng định dạng số nguyên cho `user_id`, `movie_id`, `rating`.
- Chuyển đổi giá trị `timestamp` (Unix epoch time) sang cột ngày giờ `datetime` (`YYYY-MM-DD HH:MM:SS`) phục vụ trực quan hóa và phân tích theo chuỗi thời gian.

In [6]:
cleaned_ratings = ratings_raw.copy()

# Chuyển đổi timestamp Unix sang datetime
cleaned_ratings['datetime'] = pd.to_datetime(cleaned_ratings['timestamp'], unit='s')

# Đảm bảo kiểu số nguyên
cleaned_ratings['user_id'] = cleaned_ratings['user_id'].astype(int)
cleaned_ratings['movie_id'] = cleaned_ratings['movie_id'].astype(int)
cleaned_ratings['rating'] = cleaned_ratings['rating'].astype(int)

# Sắp xếp bản ghi theo user_id và timestamp tăng dần
cleaned_ratings = cleaned_ratings.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

print("Chuẩn hóa DataFrame Ratings thành công!")
print(f"Kích thước sau xử lý: {cleaned_ratings.shape}")

Chuẩn hóa DataFrame Ratings thành công!
Kích thước sau xử lý: (1000209, 5)


## 3. Kiểm tra tính toàn vẹn của dữ liệu (Data Quality Check)
Kiểm tra số lượng giá trị thiếu (missing/null values) và bản ghi trùng lặp (duplicate rows) trong cả 3 DataFrame.

In [7]:
print("=" * 60)
print("KIỂM TRA GIÁ TRỊ THIẾU (NULL VALUES):")
print("-" * 60)
print("Users nulls:")
print(cleaned_users.isnull().sum())
print("\nMovies nulls:")
print(cleaned_movies.isnull().sum())
print("\nRatings nulls:")
print(cleaned_ratings.isnull().sum())

print("\n" + "=" * 60)
print("KIỂM TRA BẢN GHI TRÙNG LẶP (DUPLICATES):")
print("-" * 60)
print(f"Users duplicates:   {cleaned_users.duplicated().sum()}")
print(f"Movies duplicates:  {cleaned_movies.duplicated().sum()}")
print(f"Ratings duplicates: {cleaned_ratings.duplicated().sum()}")

KIỂM TRA GIÁ TRỊ THIẾU (NULL VALUES):
------------------------------------------------------------
Users nulls:
user_id            0
gender             0
gender_desc        0
age                0
age_group          0
occupation         0
occupation_name    0
zip_code           0
dtype: int64

Movies nulls:
movie_id        0
title           0
clean_title     0
release_year    0
genres          0
dtype: int64

Ratings nulls:
user_id      0
movie_id     0
rating       0
timestamp    0
datetime     0
dtype: int64

KIỂM TRA BẢN GHI TRÙNG LẶP (DUPLICATES):
------------------------------------------------------------
Users duplicates:   0
Movies duplicates:  0


Ratings duplicates: 0


## 4. Lưu kết quả làm sạch vào thư mục `data/processed/`
Xuất 3 tệp CSV đã được làm sạch:
- `cleaned_users.csv`: Dữ liệu người dùng đã chuẩn hóa tên cột, nhóm tuổi và nghề nghiệp.
- `cleaned_movies.csv`: Dữ liệu phim đã chuẩn hóa tên cột, tách năm phát hành và thể loại.
- `cleaned_ratings.csv`: Dữ liệu đánh giá đã chuẩn hóa tên cột và chuyển đổi datetime.

In [8]:
out_users_path = os.path.join(DATA_PROCESSED_DIR, 'cleaned_users.csv')
out_movies_path = os.path.join(DATA_PROCESSED_DIR, 'cleaned_movies.csv')
out_ratings_path = os.path.join(DATA_PROCESSED_DIR, 'cleaned_ratings.csv')

# Lưu DataFrame ra định dạng CSV với index=False và encoding utf-8
cleaned_users.to_csv(out_users_path, index=False, encoding='utf-8')
cleaned_movies.to_csv(out_movies_path, index=False, encoding='utf-8')
cleaned_ratings.to_csv(out_ratings_path, index=False, encoding='utf-8')

print("ĐÃ LƯU THÀNH CÔNG CÁC TỆP DỮ LIỆU:")
print(f"1. {out_users_path} ({os.path.getsize(out_users_path):,} bytes)")
print(f"2. {out_movies_path} ({os.path.getsize(out_movies_path):,} bytes)")
print(f"3. {out_ratings_path} ({os.path.getsize(out_ratings_path):,} bytes)")

ĐÃ LƯU THÀNH CÔNG CÁC TỆP DỮ LIỆU:
1. G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system\data\processed\cleaned_users.csv (274,636 bytes)
2. G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system\data\processed\cleaned_movies.csv (261,666 bytes)
3. G:\2. Học máy trong phân tích kinh doanh\FinalProject\movie-recommender-system\data\processed\cleaned_ratings.csv (42,597,937 bytes)


## 5. Hiển thị 5 dòng đầu tiên (.head()) của từng DataFrame sau khi xử lý

In [9]:
print("=" * 85)
print("1. BẢNG DỮ LIỆU NGƯỜI DÙNG ĐÃ LÀM SẠCH (CLEANED USERS) - 5 DÒNG ĐẦU TIÊN:")
print("=" * 85)
cleaned_users.head()

1. BẢNG DỮ LIỆU NGƯỜI DÙNG ĐÃ LÀM SẠCH (CLEANED USERS) - 5 DÒNG ĐẦU TIÊN:


,user_id,gender,gender_desc,age,age_group,occupation,occupation_name,zip_code
0,1,F,Female,1,Under 18,10,K-12 student,48067
1,2,M,Male,56,56+,16,self-employed,70072
2,3,M,Male,25,25-34,15,scientist,55117
3,4,M,Male,45,45-49,7,executive/managerial,02460
4,5,M,Male,25,25-34,20,writer,55455


In [10]:
print("=" * 85)
print("2. BẢNG DỮ LIỆU BỘ PHIM ĐÃ LÀM SẠCH (CLEANED MOVIES) - 5 DÒNG ĐẦU TIÊN:")
print("=" * 85)
cleaned_movies.head()

2. BẢNG DỮ LIỆU BỘ PHIM ĐÃ LÀM SẠCH (CLEANED MOVIES) - 5 DÒNG ĐẦU TIÊN:


,movie_id,title,clean_title,release_year,genres
0,1,Toy Story (1995),Toy Story,1995,Animation|Children's|Comedy
1,2,Jumanji (1995),Jumanji,1995,Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995,Comedy|Romance
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995,Comedy|Drama
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995,Comedy


In [11]:
print("=" * 85)
print("3. BẢNG DỮ LIỆU ĐÁNH GIÁ ĐÃ LÀM SẠCH (CLEANED RATINGS) - 5 DÒNG ĐẦU TIÊN:")
print("=" * 85)
cleaned_ratings.head()

3. BẢNG DỮ LIỆU ĐÁNH GIÁ ĐÃ LÀM SẠCH (CLEANED RATINGS) - 5 DÒNG ĐẦU TIÊN:


,user_id,movie_id,rating,timestamp,datetime
0,1,3186,4,978300019,2000-12-31 22:00:19
1,1,1270,5,978300055,2000-12-31 22:00:55
2,1,1721,4,978300055,2000-12-31 22:00:55
3,1,1022,5,978300055,2000-12-31 22:00:55
4,1,2340,3,978300103,2000-12-31 22:01:43
